# Transformer block 的工作流程

设输入为张量 $\mathbf{X}$（维度为 `[batch_size, sequence_length, d_model]`）

## 第一次归一化 Pre-RMSNorm

先稳住方差，防止计算爆炸。
$$\mathbf{X}_{norm1} = \text{RMSNorm}(\mathbf{X})$$
其中
$$\text{RMSNorm}(\mathbf h) = \frac{\mathbf h}{\sqrt{\frac{1}{d_{\text{model}}}\sum_{i=1}^{d_{\text{model}}} h_i^2}}$$

## 全局信息交互 Multi-Head Attention (MHA)

词与词之间交换视野，提取上下文关联。
$$\mathbf{H} = \text{MultiHeadAttention}(\mathbf{X}_{norm1})$$

### 线性投影与分头

将 $\mathbf{X}_{norm1}$ 分别乘以三个可学习的权重矩阵 $\mathbf{W}_Q$, $\mathbf{W}_K$, $\mathbf{W}_V$，得到 $\mathbf{Q}$, $\mathbf{K}$, $\mathbf{V}$。然后将 `d_model` 维度切分成 `h` 个头。例如，`d_model` 为 512，`h` 为 8，那么每个头的维度 `d_k` 就是 64。张量变形为 `[batch, h, seq_len, d_k]`。

### 旋转位置编码 RoPE

#### 空间的切片

假设我们有一个当前位置为 m 的 Query 向量 $\mathbf{q}$，它的特征维度是 `d_k`（以下简记为d）。

把这 d 维空间两两配对，切分成 d/2 个相互独立的二维子空间：
 * 子空间 1：[q_0, q_1]
 * 子空间 2：[q_2, q_3]
 * ...
 * 子空间 d/2：[q_{d-2}, q_{d-1}]

#### 乘旋转矩阵

在每个2维子空间上，我们应用一个旋转矩阵，旋转角度与位置 m 和维度 i 相关：

$$\begin{bmatrix}q'_{2i} \\ q'_{2i+1}\end{bmatrix} = \begin{bmatrix}\cos(m\theta_{i}) & -\sin(m\theta_{i}) \\ \sin(m\theta_{i}) & \cos(m\theta_{i})\end{bmatrix} \begin{bmatrix}q_{2i} \\ q_{2i+1}\end{bmatrix}$$

其中 $\theta_{i} = b^{-\frac{2i}{d}}, \quad i \in \{0,1,\dots,\frac{d}{2}-1\}$ ，$b$ 是一个常数（通常取 10000）。

#### 理论计算流程

1. 原始向量 $\mathbf{q} = [q_0, q_1, q_2, q_3, ..., q_{d-2}, q_{d-1}]$ 
2. 正负翻转向量 $\mathbf{q}_{flip} = [-q_1, q_0, -q_3, q_2, ..., -q_{d-1}, q_{d-2}]$
3. 旋转算子 
  - $\mathbf{cos} = [\cos(m\theta_0), \cos(m\theta_0), \cos(m\theta_1), \cos(m\theta_1), \cdots, \cos(m\theta_{d/2-1}), \cos(m\theta_{d/2-1})]$
  - $\mathbf{sin} = [\sin(m\theta_0), \sin(m\theta_0), \sin(m\theta_1), \sin(m\theta_1), \cdots, \sin(m\theta_{d/2-1}), \sin(m\theta_{d/2-1})]$
4. 最终旋转结果 $\mathbf{q}_{out} = \mathbf{q} \odot \mathbf{cos} + \mathbf{q}_{flip} \odot \mathbf{sin}$

#### 工业计算流程（为了计算速度）

**$q_0$ 和 $q_{d/2}$ 是一对，$q_1$ 和 $q_{d/2+1}$ 是一对，以此类推。**
1. 原始向量 $\mathbf{q} = [q_0, q_1, q_2, q_3, \cdots, q_{d-2}, q_{d-1}]$
2. 正负翻转向量 $\mathbf{q}_{flip} = [-q_{d/2}, -q_{d/2+1}, \cdots, -q_{d-1}, q_0, q_1, \cdots, q_{d-1}]$
3. 旋转算子
  - $\mathbf{cos} = [\cos(m\theta_0), \cos(m\theta_1), \cdots, \cos(m\theta_{d/2-1}), \cos(m\theta_0), \cos(m\theta_1), \cdots, \cos(m\theta_{d/2-1})]$
  - $\mathbf{sin} = [\sin(m\theta_0), \sin(m\theta_1), \cdots, \sin(m\theta_{d/2-1}), \sin(m\theta_0), \sin(m\theta_1), \cdots, \sin(m\theta_{d/2-1})]$
4. 最终旋转结果 $\mathbf{q}_{out} = \mathbf{q} \odot \mathbf{cos} + \mathbf{q}_{flip} \odot \mathbf{sin}$
    

同样的操作也应用于 Key 向量 $\mathbf{K}$。

#### 代码

In [1]:
import torch
import torch.nn as nn
class RoPE(nn.Module):
    def __init__(self, d_k, max_seq_len = 4096, b=10000):
        super().__init__()
        theta_i = 1/(b**(torch.arange(0,d_k,2).float()/d_k))  # \theta_i = b^{-\frac{2i}{d}}, \quad i \in \{0,1,\dots,\frac{d}{2}-1\}
        m = torch.arange(max_seq_len).float()  # m = [0,1,...,seq_len-1]
        m_theta_i = torch.outer(m, theta_i)  # [seq_len, d_k/2]
        cos = torch.cos(torch.cat((m_theta_i, m_theta_i), dim=-1))  # [seq_len, d_k]
        sin = torch.sin(torch.cat((m_theta_i, m_theta_i), dim=-1))  # [seq_len, d_k]
        self.register_buffer('cos', cos[None, None, :, :])  # 好处：cos和sin不需要更新参数，注册为buffer后会自动放到正确的设备上
        self.register_buffer('sin', sin[None, None, :, :])  # 扩充维度以适应后续计算

    def forward(self, x): # 适用于Q、K (V不需要位置编码!)
        # x: [batch_size, num_heads, seq_len, d_k]
        seq_len = x.shape[2]
        d_2 = x.shape[-1] // 2
        cos = self.cos[:, :, :seq_len, :]  # [1, 1, seq_len, d_k]
        sin = self.sin[:, :, :seq_len, :]
        x_first_half = x[..., :d_2]
        x_second_half = x[..., d_2:]
        x_flip = torch.cat((-x_second_half, x_first_half), dim=-1)  
        x_out = x * cos + x_flip * sin  # 旋转位置编码
        return x_out

### 缩放点积注意力与因果掩码 Causal Masking

#### Query-Key 点积计算注意力得分：
$$\mathbf{Scores}=\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}$$
得到一个 `[batch_size, h, seq_len, seq_len]` 的得分矩阵。

#### 因果掩码和 softmax 计算注意力权重
把上三角 $-\infty$ 掩码矩阵 
$$\mathbf{Mask}=\begin{bmatrix} 0 & -\infty & -\infty & \cdots \\ 0 & 0 & -\infty & \cdots \\ 0 & 0 & 0 & \cdots \\ \vdots & \vdots & \vdots & \ddots \end{bmatrix}$$
加到 $\mathbf{Scores}$ 上，遮蔽未来词并使用 softmax 计算注意力权重：
$$\mathbf{A} = \text{Softmax}(\mathbf{Scores} + \mathbf{Mask})$$
其中
$$\text{Softmax}(\mathbf{x})_i = \frac{e^{x_i}}{\sum_{j} e^{x_j}}$$

***注***：不能乘1和0的掩码，因为后面要进行softmax，$e^{-\infty}$ 才能变成0。

#### 乘以 Value 完成加权求和
最后，将注意力权重与 $\mathbf{V}$ 相乘，完成全局信息的加权坍缩：
$$\mathbf{Out} = \mathbf{A} \mathbf{V}$$

### 拼接与输出

将 `h` 个头的结果在最后一个维度上拼起来（Concat），恢复到 `d_model` 维度，得到矩阵 $\mathbf{Out'}$，最后乘以一个输出矩阵 $\mathbf{W}_O$ 进行线性映射
$$\mathbf{H} = \mathbf{Out'} \mathbf{W}_O$$

### 代码

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, d_model, max_seq_len=4096):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        # 维度属性
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_model // num_heads
        # 权重矩阵
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False) 
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        # RoPE位置编码
        self.rope = RoPE(self.d_k, max_seq_len=max_seq_len)
        # 因果掩码
        tril = torch.tril(torch.ones(max_seq_len, max_seq_len)).bool() # torch.tril (triangular lower)提取下三角元素，并把上三角元素置0
        mask = torch.zeros(max_seq_len, max_seq_len).masked_fill(~tril, float('-inf')) # 下三角元素置0，上三角元素置-inf
        self.register_buffer('mask', mask[None, None, :, :])  # [1, 1, seq_len, seq_len]

    def forward(self, x): # x: [batch_size, seq_len, d_model]
        # 线性投影与分头
        batch_size, seq_len, _ = x.shape
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)  # [batch_size, num_heads, seq_len, d_k]
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)  # 交换(1, 2)是因为之后softmax时要进行矩阵乘法(只乘后两维)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)  # 在seq_len维度上分配注意力
        # RoPE位置编码
        Q = self.rope(Q)
        K = self.rope(K)
        # 缩放点积注意力与因果掩码 Causal Masking
        ## Query-Key 点积计算注意力得分
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)  # [batch_size, num_heads, seq_len, seq_len]
        ## 因果掩码和 softmax 计算注意力权重
        Attention = F.softmax(scores + self.mask[..., :seq_len, :seq_len], dim=-1)  # [batch_size, num_heads, seq_len, seq_len]
        ## 乘以 Value 完成加权求和
        Out = torch.matmul(Attention, V)  # [batch_size, num_heads, seq_len, d_k]
        # 多头拼接并乘以输出矩阵(必须先内存连续化(contiguous)再view，否则会报错)
        Out = Out.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)  # [batch_size, seq_len, d_model]
        H = self.W_O(Out)  # [batch_size, seq_len, d_model]
        return H

## 第一次残差连接

保留原始信息，叠加全局视野
$$\mathbf{X}_1 = \mathbf{X} + \mathbf{H}$$

***注***：不要写成 $\mathbf{X}_1 = \mathbf{X}_{norm1} + \mathbf{H}$ !

## 第二次归一化

$$\mathbf{X}_{norm2} = \text{RMSNorm}(\mathbf{X}_1)$$

## 局部特征升维与非线性映射 Feed-Forward Network (FFN)

$$\mathbf{X}_{out} = \text{SwiGLU}(\mathbf{X}_{norm2})$$
其中
$$\text{SwiGLU}(x) = [\text{Swish}(xW)\odot (xV)]W_2$$

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class SwiGLU(nn.Module):
    def __init__(self,d_model,d_hidden):
        super().__init__()
        self.W = nn.Linear(d_model,d_hidden,bias=False)
        self.V = nn.Linear(d_model,d_hidden,bias=False)
        self.W2 = nn.Linear(d_hidden,d_model,bias=False)
    def forward(self,x):
        x1 = F.silu(self.W(x))
        x2 = self.V(x)
        x_out = self.W2(x1 * x2)
        return x_out

## 第二次残差连接

这就是当前 Block 的最终输出，随后原封不动地送入下一个 Block
$$\mathbf{X}_2 = \mathbf{X}_1 + \mathbf{X}_{out}$$

***注***：不要写成 $\mathbf{X}_2 = \mathbf{X}_{norm2} + \mathbf{X}_{out}$ !

## 代码拼装

In [4]:
class TransformerBlock(nn.Module):
    def __init__(self, num_heads, d_model, max_seq_len=4096, multiplier=4):
        super().__init__()
        # 用 RMSNorm 类而不是函数，因为它自动包含了可学习的缩放参数
        self.rmsnorm1 = nn.RMSNorm(d_model)  # 第一次归一化 Pre-RMSNorm
        self.rmsnorm2 = nn.RMSNorm(d_model)  # 第二次归一化
        self.attention = MultiHeadAttention(num_heads, d_model, max_seq_len=max_seq_len)
        self.ffn = SwiGLU(d_model, d_model * multiplier)
    
    def forward(self, x):
        # 第一次归一化 Pre-RMSNorm
        x_norm1 = self.rmsnorm1(x)
        # 全局信息交互 Multi-Head Attention (MHA)
        h = self.attention(x_norm1)
        # 第一次残差连接
        x1 = x + h
        # 第二次归一化
        x_norm2 = self.rmsnorm2(x1)
        # 前馈网络 Feed-Forward Network (FFN) 
        x_out = self.ffn(x_norm2)
        # 第二次残差连接
        x2 = x1 + x_out
        return x2